In [ ]:
%pip install pydeseq2 openpyxl "numpy>=1.23.5,<2.0" "fsspec>=2022.11.0" gprofiler-official


Note: you may need to restart the kernel to use updated packages.


In [2]:
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
import pandas as pd
from multiprocessing import Pool
import os

print("Reading data ~2.5 min")
counts = pd.read_excel("gs://gene_datasets/cpm_matrix_raw.xlsx")

Reading data ~2.5 min


In [3]:

target_gene_id = counts[counts['gene_name'] == 'SNHG14']['gene_id'].iloc[0]

# Transpose: genes become columns, samples become rows
cleaned = (
    counts
    .set_index('gene_id')
    .rename(index={target_gene_id: 'Target(SNHG14)'}, columns={'hSR6_1_cpm': 'Control_1(HSR6)', 'hSR6_2_cpm': 'Control_2(HSR6)'})
    .reset_index()
)

transposed = (
    cleaned
    .drop(columns=['gene_name', 'gene_biotype'])
    .set_index('gene_id')
    .T
)
transposed.index.name = 'sample'

t_counts = transposed[transposed.index.str.endswith('_count')].copy()
t_counts = t_counts.round().astype(int)

def sample_to_condition(sample):
    base = sample.replace('_count', '')
    group = base.rsplit('_', 1)[0]
    return 'control' if group == 'hSR6' else group

metadata = pd.DataFrame(
    {"condition": [sample_to_condition(s) for s in t_counts.index]},
    index=t_counts.index
)
print(metadata['condition'].value_counts())

treatments = [c for c in metadata['condition'].unique() if c != 'control']

def run_deseq2_treatment(trt):
    mask = metadata["condition"].isin(["control", trt])
    counts_sub = t_counts[mask]
    meta_sub = metadata[mask]

    dds = DeseqDataSet(counts=counts_sub, metadata=meta_sub,
                       design_factors="condition")
    dds.deseq2()

    stat_res = DeseqStats(dds, contrast=["condition", trt, "control"])
    stat_res.summary()

    results = stat_res.results_df
    results = results[results['baseMean'] > 0].copy()
    results["treatment"] = trt
    return results

condition
ZDS2        2
SP1R        2
hATF555R    2
nZF105      2
nZF93       2
hATF567     2
hATF561     2
control     2
nZFD96_3    2
nZFD96_2    2
nZFD96_1    2
nZF156      2
nZF154      2
nZF153      2
nZF151      2
nZF148      2
nZF147      2
nZF145      2
Base        2
nZF139      2
nZF81       2
nZF42       2
nZF36       2
hATF555Q    2
Name: count, dtype: int64


In [4]:
# Remove genes with zero expression across all samples
gene_mask = t_counts.sum(axis=0) > 0
t_counts = t_counts.loc[:, gene_mask]
print(f"Genes retained: {gene_mask.sum()} / {len(gene_mask)} ({(~gene_mask).sum()} zero-expression genes removed)")


Genes retained: 47942 / 78986 (31044 zero-expression genes removed)


In [5]:
pca_summary = pd.read_csv("gs://gene_datasets/pca_summary.csv")

pareto_treatments = pca_summary[pca_summary['pareto_optimal']]['sample_type'].tolist()
treatments = [t for t in pareto_treatments if t in treatments]
print(f"{len(treatments)} pareto-optimal treatments: {treatments}")

4 pareto-optimal treatments: ['hATF567', 'hATF561', 'nZF105', 'nZF139']


In [6]:
print("Run on one treatment, ~2min")
# Test with one treatment
test_result = run_deseq2_treatment(treatments[0])
test_result.head()


Run on one treatment, ~2min
Using None as control genes, passed at DeseqDataSet initialization


/tmp/ipykernel_16229/1477685312.py:40: DeprecationWarning: design_factors is deprecated and will soon be removed.Please consider providing a formulaic formula using the design argumentinstead.
  dds = DeseqDataSet(counts=counts_sub, metadata=meta_sub,
Fitting size factors...
... done in 0.01 seconds.

Fitting dispersions...
... done in 12.22 seconds.

Fitting dispersion trend curve...
... done in 1.65 seconds.

/opt/conda/miniconda3/lib/python3.10/site-packages/pydeseq2/dds.py:541: UserWarning: As the residual degrees of freedom is less than 3, the distribution of log dispersions is especially asymmetric and likely to be poorly estimated by the MAD.
  self.fit_dispersion_prior()
Fitting MAP dispersions...
... done in 18.33 seconds.

Fitting LFCs...
... done in 15.52 seconds.

Calculating cook's distance...
... done in 0.02 seconds.

Replacing 0 outlier genes.

Running Wald tests...
... done in 6.92 seconds.



Log2 fold change & Wald test p-value: condition hATF567 vs control
                   baseMean  log2FoldChange     lfcSE      stat    pvalue  \
gene_id                                                                     
ENSG00000000003   72.712122       -0.244926  0.289138 -0.847089  0.396945   
ENSG00000000005    0.000000             NaN       NaN       NaN       NaN   
ENSG00000000419  552.283816       -0.024530  0.122218 -0.200709  0.840926   
ENSG00000000457   52.912137       -0.077748  0.332036 -0.234155  0.814864   
ENSG00000000460   91.523462       -0.530080  0.264285 -2.005715  0.044887   
...                     ...             ...       ...       ...       ...   
ENSG00000310560  578.354552        0.110955  0.137337  0.807902  0.419147   
ENSG00000310566    1.039810       -0.141075  2.471372 -0.057084  0.954479   
ENSG00000310576  836.518143        0.111697  0.111649  1.000429  0.317103   
ENSG00000310577    0.548747       -2.614891  3.878381 -0.674222  0.500170   
ERCC-0000

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,treatment
gene_id,,,,,,,
ENSG00000000003,72.712122,-0.244926,0.289138,-0.847089,0.396945,0.998430,hATF567
ENSG00000000419,552.283816,-0.024530,0.122218,-0.200709,0.840926,0.998430,hATF567
ENSG00000000457,52.912137,-0.077748,0.332036,-0.234155,0.814864,0.998430,hATF567
ENSG00000000460,91.523462,-0.530080,0.264285,-2.005715,0.044887,0.770825,hATF567
ENSG00000000938,0.243322,-1.520416,5.085187,-0.298989,0.764948,NaN,hATF567


In [ ]:
# test_result[test_result['padj'] < 0.05]

In [ ]:
sc = spark.sparkContext

# Install pydeseq2 on every worker node
def install(_):
    import subprocess
    subprocess.call(["pip", "install", "pydeseq2", "numpy>=1.23.5,<2.0", "--quiet"])

sc.parallelize(range(sc.defaultParallelism), sc.defaultParallelism).foreach(install)
print(f"Installed packages on workers. Running {len(treatments)} pareto-optimal treatments: {treatments}")

# Broadcast filtered data once to all workers
t_counts_bc = sc.broadcast(t_counts)
metadata_bc = sc.broadcast(metadata)
print("Broadcasted data")

def run_spark(trt):
    import ctypes
    ctypes.CDLL("/opt/conda/miniconda3/lib/libstdc++.so.6")
    from pydeseq2.dds import DeseqDataSet
    from pydeseq2.ds import DeseqStats
    tc = t_counts_bc.value
    md = metadata_bc.value
    mask = md["condition"].isin(["control", trt])
    dds = DeseqDataSet(counts=tc[mask], metadata=md[mask], design_factors="condition")
    dds.deseq2()
    stat_res = DeseqStats(dds, contrast=["condition", trt, "control"])
    stat_res.summary()
    results = stat_res.results_df
    results = results[results['baseMean'] > 0].copy()
    results["treatment"] = trt
    print(f"Finished {trt}")
    return results.reset_index().to_dict("records")

results_dicts = sc.parallelize(treatments, len(treatments)).map(run_spark).collect()
print("Finished parallelized job")

combined = pd.concat([pd.DataFrame(r) for r in results_dicts])

combined.to_csv("gs://gene_datasets/deseq2_results.csv")
print(f"Done: {len(combined)} rows")

Installed packages on workers. Running 4 pareto-optimal treatments: ['hATF567', 'hATF561', 'nZF105', 'nZF139']
Broadcasted data


Finished parallelized job


In [14]:
combined[combined['padj']<0.05].groupby('treatment').size()

treatment
hATF561     7
hATF567    35
nZF105     45
nZF139     31
dtype: int64